In [8]:
# import openai
import os
from openai import OpenAI
import pandas as pd

# read api key from a file 
api_key = None
if os.path.exists("./openai_key"):
    with open("./openai_key", "r") as f:
        api_key = f.read().strip()
else:
    raise FileNotFoundError(f"API key file not found at {"./openai_key"}")
os.environ["OPENAI_API_KEY"] = api_key

client = OpenAI()

sys_role = """You are a SQL expert focused on compressing text columns in structured tabular data by identifying and exploiting string-based relationships between columns.

You will receive a sample CSV file that has already been loaded into a table named `df`.

Your job is to:
1. Detect all text or string columns.
2. Analyze relationships such as:
   - Shared substrings
   - Prefix/suffix matches
   - Containment (e.g., one column contains another)
   - Common tokens
   - Similar length
   - String edit distances (if supported)

You must use these relationships to **compress the data**, not just analyze them. This means:
- Removing redundant columns
- Replacing full strings with offsets, substrings, or formulas
- Merging multiple columns into one or splitting one into reusable components

---

### ✅ Output Format:

Your response **must follow this structure**:

1. **Compression Strategy**  
   - A clear explanation of the relationship used to reduce redundancy  
   - What was done to compress the data (e.g., store substring offset instead of full string)

2. **Compression SQL**  
   - SQL that transforms `df` into a compressed version, e.g., `compressed_df`  
   - Can include temporary columns for offsets, slices, etc.
   - Remove the string columns that were compressed

3. **Decompression SQL**  
   - SQL that reconstructs the original string columns from the compressed version  
   - Should produce a result equivalent to the original `df`

Use **DuckDB SQL syntax** unless specified otherwise.

"""

user_prompt = """Here is a CSV file sample:

{}

Please write SQL queries that explore the relationships between all string columns in this table. Focus on:
- Whether one string is contained in another
- Prefix/suffix relationships
- Token overlaps
- Length similarities

Use DuckDB SQL. The table is named `df`."""

def chat_completion(messages, model="gpt-4o-mini"):
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "developer", "content": sys_role},
            {
                "role": "user",
                "content": messages,
            },
        ],
    )

    print(completion.choices[0].message.content)
    return completion.choices[0].message.content






In [9]:
original_path = "./datasets_parquet/nhagar/fineweb_urls/nhagar_fineweb_urls.parquet"

df = pd.read_parquet(original_path)

sample_df = df.sample(n=1000, random_state=42)

csv_string = sample_df.to_csv(index=False)

# print(csv_string)


In [10]:
prompt = user_prompt.format(csv_string)
response = chat_completion(prompt, model="gpt-4o-mini")

To explore relationships between the string columns in the `df` table, we will create SQL queries that focus on the following aspects: containment relationships, prefix/suffix relationships, token overlaps, and length similarities. Below are the SQL queries structured according to the specified relationships.

### 1. Containment Relationships
This query checks if any of the strings in the `domain` column are contained within the `url` column.

```sql
SELECT 
    url, 
    domain, 
    CASE 
        WHEN POSITION(domain IN url) > 0 THEN 'Contained' 
        ELSE 'Not Contained' 
    END AS containment_status
FROM df;
```

### 2. Prefix/Suffix Relationships
This query checks if the `domain` is a prefix or a suffix of the `url`. 

```sql
SELECT 
    url,
    domain,
    CASE 
        WHEN url LIKE CONCAT(domain, '%') THEN 'Domain is Prefix'
        WHEN url LIKE CONCAT('%', domain) THEN 'Domain is Suffix'
        ELSE 'No Prefix/Suffix'
    END AS prefix_suffix_status
FROM df;
```

### 3.

In [14]:
import re

def extract_sql_blocks(text):
    def extract_section(title):
        # Pattern: match header followed by optional whitespace and a code block
        part = text.split(title)[1]
        query = part.split("```sql", 1)[1].split("```", 1)[0]
        return query

    compression_sql = extract_section("Compression SQL")
    decompression_sql = extract_section("Decompression SQL")

    return compression_sql, decompression_sql


In [15]:
# doc_text = """<your entire markdown string here>"""  # your original message
compress_sql, decompress_sql = extract_sql_blocks(response)

print("Compression SQL:")
print(compress_sql)

print("\nDecompression SQL:")
print(decompress_sql)

Compression SQL:

-- Create a unique domain mapping table
CREATE TABLE unique_domains AS
SELECT DISTINCT 
    domain,
    ROW_NUMBER() OVER() AS domain_id
FROM df;

-- Create the compressed table with offsets and references
CREATE TABLE compressed_df AS
SELECT 
    d.url,
    u.domain, 
    u.domain_id, 
    SUBSTR(d.url, POSITION(u.domain IN d.url) + LENGTH(u.domain) + 1) AS relative_path -- Store relative path following the domain
FROM 
    df AS d
JOIN 
    unique_domains AS u 
ON 
    POSITION(u.domain IN d.url) > 0; -- Ensure that domain is in the URL


Decompression SQL:

-- Decompress the data back to the original format
CREATE TABLE decompressed_df AS
SELECT 
    CONCAT('http://', u.domain, '/', cd.relative_path) AS url, -- Joining back the original URL
    u.domain
FROM 
    compressed_df AS cd
JOIN 
    unique_domains AS u
ON 
    cd.domain_id = u.domain_id;



In [8]:
import duckdb
con = duckdb.connect()
con.register("df", df)
print(df.head())

                                                 url                   domain
0  http://0ncemorewithfeeling.blogspot.com/2010/1...             blogspot.com
1  http://1001plus.blogspot.com/2011/11/very-shor...             blogspot.com
2  http://101bestandbrightest.com/companies/aviso...  101bestandbrightest.com
3  http://101bestandbrightest.com/companies/field...  101bestandbrightest.com
4    http://10pointsmanagement.com/the-wizard-of-oz/   10pointsmanagement.com


In [9]:
con.execute(f"""{compress_sql}""").fetchdf()
compress_df = con.execute(f"""SELECT * FROM compressed_df""").fetchdf()
print(compress_df.head)

<bound method NDFrame.head of                                                       url  \
0       http://0ncemorewithfeeling.blogspot.com/2010/1...   
1       http://1001plus.blogspot.com/2011/11/very-shor...   
2       http://101bestandbrightest.com/companies/aviso...   
3       http://101bestandbrightest.com/companies/field...   
4         http://10pointsmanagement.com/the-wizard-of-oz/   
...                                                   ...   
999995                 http://kasipkor.kz/?p=4414&lang=en   
999996  http://katm.co.uk/index.php/mother-stabs-boyfr...   
999997  http://kayser-immobiliendienst.de/en/component...   
999998                      http://kazworld.info/?p=64051   
999999  http://keeganxusmf.jaiblogs.com/2046180/not-kn...   

                            domain  domain_position  \
0                     blogspot.com               28   
1                     blogspot.com               17   
2          101bestandbrightest.com                8   
3          101bes

In [41]:
con.execute(f"""{decompress_sql}""").fetchdf()
decompressed_df = con.execute(f"""SELECT * FROM decompressed_df""").fetchdf()
print(decompressed_df.head)

<bound method NDFrame.head of                             domain  \
0                     blogspot.com   
1                     blogspot.com   
2          101bestandbrightest.com   
3          101bestandbrightest.com   
4           10pointsmanagement.com   
...                            ...   
999995                 kasipkor.kz   
999996                  katm.co.uk   
999997  kayser-immobiliendienst.de   
999998               kazworld.info   
999999                jaiblogs.com   

                                                      url  
0       http://blogspot.comorewithfeeling.blogspot.com...  
1       http://blogspot.comlus.blogspot.com/2011/11/ve...  
2       http://101bestandbrightest.comest.com/companie...  
3       http://101bestandbrightest.comest.com/companie...  
4       http://10pointsmanagement.coment.com/the-wizar...  
...                                                   ...  
999995          http://kasipkor.kzpkor.kz/?p=4414&lang=en  
999996  http://katm.co.ukm.co.uk/

In [42]:
from utils import (
    get_df,
    compare_dataframes,
    write_parquet
)

compare_dataframes(df, decompressed_df)

Differences found at these locations:
- Row 0, Column 'url': df1 = 'http://0ncemorewithfeeling.blogspot.com/2010/11/shopping-at-home.html', df2 = 'http://blogspot.comorewithfeeling.blogspot.com/2010/11/shopping-at-home.html'
- Row 1, Column 'url': df1 = 'http://1001plus.blogspot.com/2011/11/very-short-marathon.html', df2 = 'http://blogspot.comlus.blogspot.com/2011/11/very-short-marathon.html'
- Row 2, Column 'url': df1 = 'http://101bestandbrightest.com/companies/avisolve/', df2 = 'http://101bestandbrightest.comest.com/companies/avisolve/'
- Row 3, Column 'url': df1 = 'http://101bestandbrightest.com/companies/fieldedge/', df2 = 'http://101bestandbrightest.comest.com/companies/fieldedge/'
- Row 4, Column 'url': df1 = 'http://10pointsmanagement.com/the-wizard-of-oz/', df2 = 'http://10pointsmanagement.coment.com/the-wizard-of-oz/'
- Row 5, Column 'url': df1 = 'http://123-free-download.com/download/network/network-inventory-advisor-for-mac/356412.html', df2 = 'http://123-free-download.comoa

KeyboardInterrupt: 